In [ ]:
import os
import pickle
import math
import copy
import random
import json
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

In [ ]:
random.seed(18)
np.random.seed(18)
torch.manual_seed(18)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(18)
    torch.backends.cudnn.deterministic = True

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

os.makedirs('./checkpoints', exist_ok=True)

In [ ]:
DATA_DIR = "./data/processed"
os.makedirs(DATA_DIR, exist_ok=True)

In [ ]:
class AmplitudeTokenizer:
    def __init__(self, index_pool_size=100, special_symbols=None, to_replace=True):
        self.index_pool_size = index_pool_size
        self.to_replace = to_replace

        if special_symbols is None:
            self.special_symbols = ['<PAD>', '<UNK>', '<BOS>', '<EOS>', '<SEP>', '<TERM0>', '<TERM1>']
        else:
            self.special_symbols = special_symbols

        self.index_pool = [f"INDEX_{i}" for i in range(index_pool_size)]
        self.particle_index_pool = [f"PINDEX_{i}" for i in range(index_pool_size)]

        self.pattern_particle = re.compile(r'(?P<prefix>\b(?:\w+_)?)?(?P<target>[ijkl]_\d+\b)')
        self.pattern_num_123 = re.compile(r'\b(?![psijkl]_)(?!MOMENTUM_)(?!\w+_\w+_)\w+_\d+\b')
        self.pattern_momentum = re.compile(r'[ps]_(\d+)')

    @staticmethod
    def remove_whitespace(text):
        return re.sub(r'\s+', '', text)

    @staticmethod
    def split_expression(text):
        return re.split(r' ', text)

    def normalize_indices(self, text):
        if not self.to_replace:
            return text

        text = self.remove_whitespace(text)

        text = re.sub(r'p_(\d+)', r'MOMENTUM_\1', text)
        text = re.sub(r's_(\d+)', r'MOMENTUM_\1', text)

        text = text.replace('\\\\', '\\').replace('\\', r' \ ').replace('%', ' % ')

        index_matches = list(OrderedDict.fromkeys(
            self.pattern_num_123.findall(text)
        ))

        index_iter = iter(self.index_pool)
        index_mapping = {}
        for match in index_matches:
            try:
                index_mapping[match] = next(index_iter)
            except StopIteration:
                raise RuntimeError(f"Index pool exhausted. Increase index_pool_size (currently {self.index_pool_size})")

        for old, new in sorted(index_mapping.items(), key=lambda x: len(x[0]), reverse=True):
            text = text.replace(old, new)

        particle_matches = list(OrderedDict.fromkeys(
            m.group('target')
            for m in sorted(self.pattern_particle.finditer(text), key=lambda m: m.start())
        ))

        particle_iter = iter(self.particle_index_pool)
        particle_mapping = {}
        for match in particle_matches:
            try:
                particle_mapping[match] = next(particle_iter)
            except StopIteration:
                raise RuntimeError("Particle index pool exhausted. Increase index_pool_size")

        for old, new in sorted(particle_mapping.items(), key=lambda x: len(x[0]), reverse=True):
            text = text.replace(old, new)

        return text

    def tokenize_amplitude(self, amplitude_text):
        if self.to_replace:
            text = self.normalize_indices(amplitude_text)
        else:
            text = amplitude_text

        text = self.remove_whitespace(text)

        text = text.replace('\\\\', '\\').replace('\\', r' \ ').replace('%', ' % ')
        text = text.replace("(*)", " CONJ ")
        text = text.replace("(theta_W)", "_theta_W")

        for symbol in ['/', '+', '-', '*', ',', '^', '%', '}', '(', ')', '=', '[', ']']:
            text = text.replace(symbol, f' {symbol} ')

        text = text.replace("_PINDEX", "_ PINDEX").replace("_INDEX", "_ INDEX")
        text = text.replace("reg_prop", " reg_prop ")

        text = re.sub(r' {2,}', ' ', text)

        tokens = [t for t in self.split_expression(text) if t]
        return tokens

    def tokenize_squared(self, squared_text):
        if self.to_replace:
            text = self.normalize_indices(squared_text)
        else:
            text = squared_text

        text = self.remove_whitespace(text)
        text = text.replace("(theta_W)", "_theta_W")

        for symbol in ['/', '+', '-', '*', ',', '^', '%', '}', '(', ')', '=', '[', ']']:
            text = text.replace(symbol, f' {symbol} ')

        text = re.sub(r'\bm_(\w+)\b', r' m_\1 ', text)

        text = re.sub(r'\bs_(\d{2,})\b', r' s_\1 ', text)

        text = text.replace("reg_prop", " reg_prop ")
        text = re.sub(r' {2,}', ' ', text)

        tokens = [t for t in self.split_expression(text) if t]
        return tokens

In [ ]:
import pickle

class Vocabulary:
    def __init__(self, tokens, special_symbols,
                 pad_idx=0, unk_idx=1, bos_idx=2, eos_idx=3, sep_idx=4, term_idx=[5,6]):
        tokens = list(tokens)
        for special in special_symbols:
            if special in tokens:
                tokens.remove(special)
        self.token_list = special_symbols + tokens
        self.token_to_idx = {token: idx for idx, token in enumerate(self.token_list)}
        self.idx_to_token = {idx: token for token, idx in self.token_to_idx.items()}
        self.pad_idx = pad_idx
        self.unk_idx = unk_idx
        self.bos_idx = bos_idx
        self.eos_idx = eos_idx
        self.sep_idx = sep_idx
        self.term_idx = term_idx
        self.pad_tok = special_symbols[pad_idx]
        self.unk_tok = special_symbols[unk_idx]
        self.bos_tok = special_symbols[bos_idx]
        self.eos_tok = special_symbols[eos_idx]
        self.sep_tok = special_symbols[sep_idx]
        self.special_indices = set(self.token_to_idx[sym] for sym in special_symbols)

    def encode(self, tokens):
        return [self.token_to_idx.get(token, self.unk_idx) for token in tokens]

    def decode(self, indices, include_special=True):
        if include_special:
            return [self.idx_to_token.get(idx, self.unk_tok) for idx in indices]
        else:
            return [
                self.idx_to_token.get(idx, self.unk_tok)
                for idx in indices
                if idx not in self.special_indices or idx == self.sep_idx
            ]

    def __len__(self):
        return len(self.token_list)

    def __getitem__(self, item):
        if isinstance(item, int):
            return self.idx_to_token.get(item, self.unk_tok)
        return self.token_to_idx.get(item, self.unk_idx)

    def tokens(self):
        return self.token_list

DATA_PKL = './data/processed/processed_data.pkl'

with open(DATA_PKL, 'rb') as f:
    data = pickle.load(f)

qed_train = data['qed']['train']
qed_val   = data['qed']['val']
qed_test  = data['qed']['test']
qed_src_vocab = data['qed']['src_vocab']
qed_tgt_vocab = data['qed']['tgt_vocab']

qcd_train = data['qcd']['train']
qcd_val   = data['qcd']['val']
qcd_test  = data['qcd']['test']
qcd_src_vocab = data['qcd']['src_vocab']
qcd_tgt_vocab = data['qcd']['tgt_vocab']

print(f"QED — Train: {len(qed_train)}, Val: {len(qed_val)}, Test: {len(qed_test)}")
print(f"QCD — Train: {len(qcd_train)}, Val: {len(qcd_val)}, Test: {len(qcd_test)}")
print(f"\nQED src vocab: {len(qed_src_vocab)} tokens")
print(f"QED tgt vocab: {len(qed_tgt_vocab)} tokens")
print(f"QCD src vocab: {len(qcd_src_vocab)} tokens")
print(f"QCD tgt vocab: {len(qcd_tgt_vocab)} tokens")

sample = qed_train[0]
print(f"\nSample keys: {list(sample.keys())}")
print(f"Amp tokens (first 10): {sample['amp_tokens'][:10]}")
print(f"Sq tokens (first 10): {sample['sq_tokens'][:10]}")

print(f"\nQED src vocab special tokens:")
print(f"  PAD={qed_src_vocab.pad_idx}, UNK={qed_src_vocab.unk_idx}, "
      f"BOS={qed_src_vocab.bos_idx}, EOS={qed_src_vocab.eos_idx}, "
      f"SEP={qed_src_vocab.sep_idx}")
print(f"  First 10 tokens: {qed_src_vocab.token_list[:10]}")

QED — Train: 288, Val: 36, Test: 36
QCD — Train: 187, Val: 23, Test: 24

QED src vocab: 125 tokens
QED tgt vocab: 43 tokens
QCD src vocab: 138 tokens
QCD tgt vocab: 67 tokens

Sample keys: ['interaction', 'diagram', 'amplitude', 'squared_amplitude', 'source_file', 'line_num', 'amplitude_norm', 'squared_norm', 'amp_tokens', 'sq_tokens', 'amp_len', 'sq_len']
Amp tokens (first 10): ['-', '1', '/', '6', '*', 'i', '*', 'e', '^', '2']
Sq tokens (first 10): ['1', '/', '36', '*', 'e', '^', '4', '*', '(', '16']

QED src vocab special tokens:
  PAD=0, UNK=1, BOS=2, EOS=3, SEP=4
  First 10 tokens: ['<PAD>', '<UNK>', '<BOS>', '<EOS>', '<SEP>', '<TERM0>', '<TERM1>', '%', '(', ')']


Unified Vocabulary

In [ ]:
class UnifiedVocabulary:
    def __init__(self):
        self.token_to_idx = {}
        self.idx_to_token = {}
        self.token_list = []

    def build_from_vocabs(self, src_vocab, tgt_vocab):
        special_tokens = ['<PAD>', '<UNK>', '<BOS>', '<EOS>', '<SEP>', '<PRED>']

        src_specials = set()
        if hasattr(src_vocab, 'special_indices'):
            for idx in src_vocab.special_indices:
                tok = src_vocab.idx_to_token[idx]
                if tok not in special_tokens:
                    special_tokens.append(tok)
                src_specials.add(tok)

        for i, tok in enumerate(special_tokens):
            self.token_to_idx[tok] = i
            self.idx_to_token[i] = tok
        self.token_list = list(special_tokens)

        idx = len(special_tokens)

        src_added = 0
        for tok in src_vocab.token_list:
            if tok not in self.token_to_idx:
                self.token_to_idx[tok] = idx
                self.idx_to_token[idx] = tok
                self.token_list.append(tok)
                idx += 1
                src_added += 1

        tgt_added = 0
        for tok in tgt_vocab.token_list:
            if tok not in self.token_to_idx:
                self.token_to_idx[tok] = idx
                self.idx_to_token[idx] = tok
                self.token_list.append(tok)
                idx += 1
                tgt_added += 1

        print(f"  Unified vocab size: {len(self.token_to_idx)}")
        print(f"    From src: {src_added} unique tokens added")
        print(f"    From tgt: {tgt_added} unique tokens added")
        print(f"    Shared tokens: {len(src_vocab.token_list) + len(tgt_vocab.token_list) - len(self.token_to_idx)} overlap")

    @property
    def pad_idx(self):
        return self.token_to_idx['<PAD>']

    @property
    def unk_idx(self):
        return self.token_to_idx['<UNK>']

    @property
    def bos_idx(self):
        return self.token_to_idx['<BOS>']

    @property
    def eos_idx(self):
        return self.token_to_idx['<EOS>']

    @property
    def sep_idx(self):
        return self.token_to_idx['<SEP>']

    @property
    def pred_idx(self):
        return self.token_to_idx['<PRED>']

    def encode(self, tokens):
        return [self.token_to_idx.get(t, self.unk_idx) for t in tokens]

    def decode(self, ids):
        result = []
        for idx in ids:
            tok = self.idx_to_token.get(idx, '<UNK>')
            if tok == '<EOS>':
                break
            if tok not in ('<PAD>', '<BOS>', '<PRED>', '<SEP>'):
                result.append(tok)
        return ' '.join(result)

    def __len__(self):
        return len(self.token_to_idx)

    def __getitem__(self, item):
        if isinstance(item, int):
            return self.idx_to_token.get(item, '<UNK>')
        return self.token_to_idx.get(item, self.unk_idx)

print("Building unified QED vocabulary:")
qed_vocab = UnifiedVocabulary()
qed_vocab.build_from_vocabs(qed_src_vocab, qed_tgt_vocab)

print(f"\nBuilding unified QCD vocabulary:")
qcd_vocab = UnifiedVocabulary()
qcd_vocab.build_from_vocabs(qcd_src_vocab, qcd_tgt_vocab)

print(f"\n--- Verification ---")
print(f"QED unified vocab: {len(qed_vocab)} tokens")
print(f"  PAD={qed_vocab.pad_idx}, UNK={qed_vocab.unk_idx}, "
      f"BOS={qed_vocab.bos_idx}, EOS={qed_vocab.eos_idx}, "
      f"SEP={qed_vocab.sep_idx}, PRED={qed_vocab.pred_idx}")
print(f"  First 10 tokens: {qed_vocab.token_list[:10]}")

print(f"\nQCD unified vocab: {len(qcd_vocab)} tokens")
print(f"  PAD={qcd_vocab.pad_idx}, UNK={qcd_vocab.unk_idx}, "
      f"BOS={qcd_vocab.bos_idx}, EOS={qcd_vocab.eos_idx}, "
      f"SEP={qcd_vocab.sep_idx}, PRED={qcd_vocab.pred_idx}")

sample = qed_train[0]
test_tokens = sample['amp_tokens'][:5]
encoded = qed_vocab.encode(test_tokens)
decoded = qed_vocab.decode(encoded)
print(f"\nEncoding test:")
print(f"  Tokens:  {test_tokens}")
print(f"  Encoded: {encoded}")
print(f"  Decoded: {decoded}")

Building unified QED vocabulary:
  Unified vocab size: 136
    From src: 118 unique tokens added
    From tgt: 10 unique tokens added
    Shared tokens: 32 overlap

Building unified QCD vocabulary:
  Unified vocab size: 175
    From src: 131 unique tokens added
    From tgt: 36 unique tokens added
    Shared tokens: 30 overlap

--- Verification ---
QED unified vocab: 136 tokens
  PAD=0, UNK=1, BOS=2, EOS=3, SEP=4, PRED=5
  First 10 tokens: ['<PAD>', '<UNK>', '<BOS>', '<EOS>', '<SEP>', '<PRED>', '<TERM0>', '<TERM1>', '%', '(']

QCD unified vocab: 175 tokens
  PAD=0, UNK=1, BOS=2, EOS=3, SEP=4, PRED=5

Encoding test:
  Tokens:  ['-', '1', '/', '6', '*']
  Encoded: [14, 16, 15, 21, 11]
  Decoded: - 1 / 6 *


Dataset classes
# PretrainDataset: splits amplitude into prefix/suffix for JEPA
# FinetuneDataset: pairs amplitude with squared amplitude

In [ ]:
class PretrainDataset(Dataset):
    def __init__(self, data_list, vocab):
        self.data = data_list
        self.vocab = vocab

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        tokens = self.data[idx]['amp_tokens']

        n = len(tokens)
        lo = max(1, int(n * 0.4))
        hi = min(n - 1, int(n * 0.6))
        if lo >= hi:
            split = max(1, n // 2)
        else:
            split = random.randint(lo, hi)

        prefix_tokens = tokens[:split]
        suffix_tokens = tokens[split:]

        prefix_ids = ([self.vocab.bos_idx]
                      + self.vocab.encode(prefix_tokens)
                      + [self.vocab.eos_idx])
        suffix_ids = ([self.vocab.bos_idx]
                      + self.vocab.encode(suffix_tokens)
                      + [self.vocab.eos_idx])

        return (torch.tensor(prefix_ids, dtype=torch.long),
                torch.tensor(suffix_ids, dtype=torch.long))


class FinetuneDataset(Dataset):
    def __init__(self, data_list, vocab):
        self.data = data_list
        self.vocab = vocab

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        amp_tokens = item['amp_tokens']
        sq_tokens = item['sq_tokens']

        amp_ids = self.vocab.encode(amp_tokens)
        sq_ids = self.vocab.encode(sq_tokens)

        full_ids = ([self.vocab.bos_idx]
                    + amp_ids
                    + [self.vocab.sep_idx]
                    + sq_ids
                    + [self.vocab.eos_idx])

        sep_pos = 1 + len(amp_ids)

        amp_only = ([self.vocab.bos_idx]
                    + amp_ids
                    + [self.vocab.eos_idx])

        sq_only = ([self.vocab.bos_idx]
                   + sq_ids
                   + [self.vocab.eos_idx])

        return {
            'full': torch.tensor(full_ids, dtype=torch.long),
            'sep_pos': sep_pos,
            'amp': torch.tensor(amp_only, dtype=torch.long),
            'sq_amp': torch.tensor(sq_only, dtype=torch.long),
        }

In [ ]:
# Collating the data

def collate_pretrain(batch):
    """Pad view1 and view2 separately."""
    view1 = [item[0] for item in batch]
    view2 = [item[1] for item in batch]
    v1_padded = pad_sequence(view1, batch_first=True, padding_value=0)
    v2_padded = pad_sequence(view2, batch_first=True, padding_value=0)
    return v1_padded, v2_padded


def collate_finetune(batch):
    """Pad full, amp, sq_amp separately. Collect sep_pos."""
    full_seqs = [item['full'] for item in batch]
    amp_seqs  = [item['amp'] for item in batch]
    sq_seqs   = [item['sq_amp'] for item in batch]
    sep_positions = torch.tensor([item['sep_pos'] for item in batch], dtype=torch.long)

    full_padded = pad_sequence(full_seqs, batch_first=True, padding_value=0)
    amp_padded  = pad_sequence(amp_seqs, batch_first=True, padding_value=0)
    sq_padded   = pad_sequence(sq_seqs, batch_first=True, padding_value=0)

    return {
        'full': full_padded,
        'sep_pos': sep_positions,
        'amp': amp_padded,
        'sq_amp': sq_padded,
    }

LM JEPA MODEL (QED)

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float()
                        * -(math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return self.dropout(x + self.pe[:, :x.size(1)])

In [ ]:
class DecoderOnlyLMJEPA(nn.Module):
    def __init__(
        self,
        vocab_size,
        d_model=256,
        nhead=8,
        num_layers=6,
        dim_feedforward=512,
        dropout=0.1,
        max_len=4096,
        num_pred_tokens=2,
        pad_idx=0,
    ):
        super().__init__()
        self.d_model = d_model
        self.num_pred_tokens = num_pred_tokens
        self.pad_idx = pad_idx
        self.vocab_size = vocab_size

        self.token_embed = nn.Embedding(vocab_size, d_model, padding_idx=pad_idx)
        self.pos_enc = PositionalEncoding(d_model, max_len=max_len, dropout=dropout)

        layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,
            norm_first=True,
            activation='gelu',
        )
        self.transformer = nn.TransformerEncoder(layer, num_layers=num_layers)

        self.pred_token_embed = nn.Parameter(
            torch.randn(1, num_pred_tokens, d_model) * 0.02
        )

        self.output_norm = nn.LayerNorm(d_model)
        self.output_proj = nn.Linear(d_model, vocab_size, bias=False)

        self.embed_norm = nn.LayerNorm(d_model)

        self.output_proj.weight = self.token_embed.weight

        self._init_weights()

    def _init_weights(self):
        nn.init.normal_(self.token_embed.weight, std=0.02)
        for name, p in self.transformer.named_parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)
            elif 'bias' in name:
                nn.init.zeros_(p)

    def _causal_mask(self, seq_len, device):
        return nn.Transformer.generate_square_subsequent_mask(
            seq_len, device=device
        )

    def _block_causal_mask(self, len1, len2, device):
        total = len1 + len2
        mask = torch.full((total, total), float('-inf'), device=device)

        mask[:len1, :len1] = self._causal_mask(len1, device)
        mask[len1:, len1:] = self._causal_mask(len2, device)

        return mask

    def _last_non_pad_idx(self, token_ids):
        non_pad = (token_ids != self.pad_idx).long()
        lengths = non_pad.sum(dim=1) - 1
        return lengths.clamp(min=0)

    def _forward_transformer(self, embeds, mask=None, pad_mask=None):
        return self.transformer(
            embeds,
            mask=mask,
            src_key_padding_mask=pad_mask,
        )

    def get_embedding(self, token_ids):
        seq_len = token_ids.size(1)
        pad_mask = (token_ids == self.pad_idx)

        x = self.token_embed(token_ids) * math.sqrt(self.d_model)
        x = self.pos_enc(x)

        hidden = self._forward_transformer(
            x,
            mask=self._causal_mask(seq_len, token_ids.device),
            pad_mask=pad_mask,
        )

        last_idx = self._last_non_pad_idx(token_ids)
        batch_idx = torch.arange(token_ids.size(0), device=token_ids.device)
        embedding = hidden[batch_idx, last_idx]

        return self.embed_norm(embedding)

    def predict_embedding(self, token_ids):
        batch_size = token_ids.size(0)

        x = self.token_embed(token_ids) * math.sqrt(self.d_model)
        x = self.pos_enc(x)

        pred_embeds = self.pred_token_embed.expand(batch_size, -1, -1)
        x = torch.cat([x, pred_embeds], dim=1)

        total_len = x.size(1)

        pad_mask = torch.cat([
            (token_ids == self.pad_idx),
            torch.zeros(batch_size, self.num_pred_tokens,
                       dtype=torch.bool, device=token_ids.device),
        ], dim=1)

        causal = self._causal_mask(total_len, token_ids.device)

        hidden = self._forward_transformer(x, mask=causal, pad_mask=pad_mask)

        pred_output = hidden[:, -1, :]

        return self.embed_norm(pred_output)

    def get_two_view_embeddings(self, view1_ids, view2_ids):
        batch_size = view1_ids.size(0)
        len1 = view1_ids.size(1)
        len2 = view2_ids.size(1)

        combined = torch.cat([view1_ids, view2_ids], dim=1)

        x = self.token_embed(combined) * math.sqrt(self.d_model)
        x = self.pos_enc(x)

        block_mask = self._block_causal_mask(len1, len2, combined.device)

        pad_mask = (combined == self.pad_idx)

        hidden = self._forward_transformer(x, mask=block_mask, pad_mask=pad_mask)

        last1 = self._last_non_pad_idx(view1_ids)
        last2 = self._last_non_pad_idx(view2_ids) + len1

        batch_idx = torch.arange(batch_size, device=combined.device)
        emb1 = self.embed_norm(hidden[batch_idx, last1])
        emb2 = self.embed_norm(hidden[batch_idx, last2])

        return emb1, emb2

    def forward(self, token_ids):
        seq_len = token_ids.size(1)
        pad_mask = (token_ids == self.pad_idx)

        x = self.token_embed(token_ids) * math.sqrt(self.d_model)
        x = self.pos_enc(x)

        hidden = self._forward_transformer(
            x,
            mask=self._causal_mask(seq_len, token_ids.device),
            pad_mask=pad_mask,
        )

        hidden = self.output_norm(hidden)
        logits = self.output_proj(hidden)
        return logits

Loss Function

In [ ]:
def compute_llm_loss(logits, token_ids, sep_positions, pad_idx=0):

    batch_size, seq_len, vocab_size = logits.size()

    shift_logits = logits[:, :-1, :]
    shift_targets = token_ids[:, 1:]

    loss_mask = torch.zeros_like(shift_targets, dtype=torch.float)
    for i in range(batch_size):
        start = sep_positions[i].item()
        loss_mask[i, start:] = 1.0

    loss_mask = loss_mask * (shift_targets != pad_idx).float()

    ce = F.cross_entropy(
        shift_logits.reshape(-1, vocab_size),
        shift_targets.reshape(-1),
        reduction='none',
    ).reshape(batch_size, -1)

    masked_loss = (ce * loss_mask).sum()
    num_tokens = loss_mask.sum().clamp(min=1)

    return masked_loss / num_tokens


def compute_jepa_loss(pred_emb, target_emb):

    cos_sim = F.cosine_similarity(pred_emb, target_emb, dim=-1)
    return (1 - cos_sim).mean()

Combined Loss

In [ ]:
def pretrain_one_epoch(model, loader, optimizer, device):

    model.train()
    total_loss = 0.0
    n_batches = 0

    for view1, view2 in loader:
        view1, view2 = view1.to(device), view2.to(device)

        with torch.no_grad():
            target_emb = model.get_embedding(view2)

        pred_emb = model.predict_embedding(view1)

        loss = compute_jepa_loss(pred_emb, target_emb)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()
        n_batches += 1

    return total_loss / max(n_batches, 1)


In [ ]:
def finetune_one_epoch(model, loader, optimizer, lambda_jepa,
                        jepa_dropout=0.5, device='cuda'):

    model.train()
    total_loss = 0.0
    total_llm = 0.0
    total_jepa = 0.0
    n_batches = 0

    for batch in loader:
        full = batch['full'].to(device)
        sep_pos = batch['sep_pos'].to(device)
        amp = batch['amp'].to(device)
        sq_amp = batch['sq_amp'].to(device)

        logits = model(full)
        loss_llm = compute_llm_loss(logits, full, sep_pos, model.pad_idx)

        batch_size = amp.size(0)
        n_jepa = max(1, int(batch_size * (1 - jepa_dropout)))
        jepa_idx = torch.randperm(batch_size, device=device)[:n_jepa]

        amp_sub = amp[jepa_idx]
        sq_sub = sq_amp[jepa_idx]

        pred_emb = model.predict_embedding(amp_sub)

        with torch.no_grad():
            target_emb = model.get_embedding(sq_sub)

        loss_jepa = compute_jepa_loss(pred_emb, target_emb)

        loss = loss_llm + lambda_jepa * loss_jepa

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()
        total_llm += loss_llm.item()
        total_jepa += loss_jepa.item()
        n_batches += 1

    n = max(n_batches, 1)
    return total_loss / n, total_llm / n, total_jepa / n


@torch.no_grad()
def validate(model, loader, lambda_jepa, device='cuda'):

    model.eval()
    total_loss = 0.0
    total_llm = 0.0
    total_jepa = 0.0
    n_batches = 0

    for batch in loader:
        full = batch['full'].to(device)
        sep_pos = batch['sep_pos'].to(device)
        amp = batch['amp'].to(device)
        sq_amp = batch['sq_amp'].to(device)

        logits = model(full)
        loss_llm = compute_llm_loss(logits, full, sep_pos, model.pad_idx)

        pred_emb = model.predict_embedding(amp)
        target_emb = model.get_embedding(sq_amp)
        loss_jepa = compute_jepa_loss(pred_emb, target_emb)

        loss = loss_llm + lambda_jepa * loss_jepa

        total_loss += loss.item()
        total_llm += loss_llm.item()
        total_jepa += loss_jepa.item()
        n_batches += 1

    n = max(n_batches, 1)
    return total_loss / n, total_llm / n, total_jepa / n

Model Evaluation

In [ ]:
@torch.no_grad()
def evaluate_model(model, loader, vocab, max_gen_len=300, device='cuda'):
    model.eval()

    total_correct = 0
    total_tokens = 0
    total_ce = 0.0
    total_cos = 0.0
    total_samples = 0

    for batch in loader:
        full = batch['full'].to(device)
        sep_pos = batch['sep_pos'].to(device)
        amp = batch['amp'].to(device)
        sq_amp = batch['sq_amp'].to(device)
        batch_size = full.size(0)

        logits = model(full)

        preds = logits[:, :-1, :].argmax(dim=-1)
        targets = full[:, 1:]

        for i in range(batch_size):
            start = sep_pos[i].item()
            seq_len = targets.size(1)
            for t in range(start, seq_len):
                if targets[i, t] == vocab.pad_idx:
                    break
                total_tokens += 1
                if preds[i, t] == targets[i, t]:
                    total_correct += 1
                log_prob = F.log_softmax(logits[i, t, :], dim=-1)
                total_ce -= log_prob[targets[i, t]].item()

        pred_emb = model.predict_embedding(amp)
        target_emb = model.get_embedding(sq_amp)
        cos = F.cosine_similarity(pred_emb, target_emb, dim=-1)
        total_cos += cos.sum().item()
        total_samples += batch_size

    token_accuracy = total_correct / max(total_tokens, 1)
    perplexity = math.exp(min(total_ce / max(total_tokens, 1), 100))
    jepa_cosine = total_cos / max(total_samples, 1)

    exact_matches = 0
    total_seqs = 0

    for batch in loader:
        amp = batch['amp'].to(device)
        sq_amp = batch['sq_amp'].to(device)
        batch_size = amp.size(0)

        for i in range(batch_size):
            amp_seq = amp[i]
            amp_tokens = amp_seq[amp_seq != vocab.pad_idx].tolist()
            if amp_tokens[-1] == vocab.eos_idx:
                amp_tokens = amp_tokens[:-1]
            input_ids = amp_tokens + [vocab.sep_idx]
            input_tensor = torch.tensor([input_ids], dtype=torch.long, device=device)

            for _ in range(max_gen_len):
                logits = model(input_tensor)
                next_token = logits[0, -1, :].argmax().item()
                input_tensor = torch.cat([
                    input_tensor,
                    torch.tensor([[next_token]], device=device)
                ], dim=1)
                if next_token == vocab.eos_idx:
                    break

            full_gen = input_tensor[0].tolist()
            try:
                sep_loc = full_gen.index(vocab.sep_idx)
            except ValueError:
                sep_loc = 0
            generated = []
            for t in full_gen[sep_loc + 1:]:
                if t == vocab.eos_idx or t == vocab.pad_idx:
                    break
                generated.append(t)

            sq_seq = sq_amp[i]
            gt = []
            for t in sq_seq.tolist():
                if t in (vocab.bos_idx, vocab.pad_idx):    # <-- FIXED: was vocab.sos_idx
                    continue
                if t == vocab.eos_idx:
                    break
                gt.append(t)

            if generated == gt:
                exact_matches += 1
            total_seqs += 1

    seq_exact_match = exact_matches / max(total_seqs, 1)

    results = {
        'token_accuracy': token_accuracy,
        'sequence_exact_match': seq_exact_match,
        'perplexity': perplexity,
        'jepa_cosine_sim': jepa_cosine,
    }

    return results


def print_results(results, label=""):
    if label:
        print(f"\nResults - {label}")
    else:
        print("\nResults")

    print("Token Accuracy:", round(results['token_accuracy'], 4),
          f"({round(results['token_accuracy']*100, 1)}%)")
    print("Sequence Exact Match:", round(results['sequence_exact_match'], 4),
          f"({round(results['sequence_exact_match']*100, 1)}%)")
    print("Perplexity:", round(results['perplexity'], 2))
    print("JEPA Cosine Similarity:", round(results['jepa_cosine_sim'], 4))

Config File (Hyperparameters)

d_model=256, heads=8, layers=6

In [ ]:
# Architecture
D_MODEL = 256
NHEAD = 8
NUM_LAYERS = 6
DIM_FEEDFORWARD = 512
NUM_PRED_TOKENS = 2
DROPOUT = 0.1
MAX_LEN = 4096

# Training
BATCH_SIZE = 32
PRETRAIN_EPOCHS = 25
FINETUNE_EPOCHS = 50
PRETRAIN_LR = 3e-4
FINETUNE_LR = 1e-4
WEIGHT_DECAY = 0.01
LAMBDA_JEPA = 1.0
JEPA_DROPOUT = 0.5

# Early stopping
PATIENCE = 10

Model Training

Pretraining

In [ ]:
def create_model(vocab, device=device):
    model = DecoderOnlyLMJEPA(
        vocab_size=len(vocab),
        d_model=D_MODEL,
        nhead=NHEAD,
        num_layers=NUM_LAYERS,
        dim_feedforward=DIM_FEEDFORWARD,
        dropout=DROPOUT,
        max_len=MAX_LEN,
        num_pred_tokens=NUM_PRED_TOKENS,
        pad_idx=vocab.pad_idx,
    ).to(device)

    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  Model created — {n_params:,} trainable parameters")
    return model


def run_pretraining(model, train_data, vocab, model_name='QED', device=device):
    print(f"\n{'='*60}")
    print(f"  PHASE 1: JEPA PRETRAINING — {model_name}")
    print(f"{'='*60}")

    pretrain_loader = DataLoader(
        PretrainDataset(train_data, vocab),
        batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_pretrain,
    )

    optimizer = torch.optim.AdamW(
        model.parameters(), lr=PRETRAIN_LR, weight_decay=WEIGHT_DECAY
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=PRETRAIN_EPOCHS, eta_min=1e-6
    )

    pretrain_history = []

    for epoch in range(1, PRETRAIN_EPOCHS + 1):
        loss = pretrain_one_epoch(model, pretrain_loader, optimizer, device)
        scheduler.step()
        pretrain_history.append(loss)

        if epoch % 5 == 0 or epoch == 1:
            print(f"    Epoch {epoch:3d}/{PRETRAIN_EPOCHS} | "
                  f"JEPA Loss: {loss:.4f} | "
                  f"LR: {scheduler.get_last_lr()[0]:.2e}")

    save_path = f'./checkpoints/{model_name.lower()}_pretrained.pt'
    torch.save(model.state_dict(), save_path)
    print(f"\n  ✓ Pretrained model saved to {save_path}")
    print(f"  ✓ Final JEPA loss: {pretrain_history[-1]:.4f}")

    return pretrain_history



def run_evaluation(model, test_data, vocab, model_name='QED', device=device):

    print(f"\n{'='*60}")
    print(f"  EVALUATION — {model_name}")
    print(f"{'='*60}")

    test_loader = DataLoader(
        FinetuneDataset(test_data, vocab),
        batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_finetune,
    )

    results = evaluate_model(model, test_loader, vocab, device=device)
    print_results(results, label=f"[{model_name}]")

    return results

Finetuning

In [ ]:
def run_finetuning(model, train_data, val_data, vocab,
                   lambda_jepa=LAMBDA_JEPA, model_name='QED', device=device):

    print(f"\n{'='*60}")
    print(f"  PHASE 2: FINE-TUNING — {model_name} (λ={lambda_jepa})")
    print(f"{'='*60}")

    finetune_loader = DataLoader(
        FinetuneDataset(train_data, vocab),
        batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_finetune,
    )
    val_loader = DataLoader(
        FinetuneDataset(val_data, vocab),
        batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_finetune,
    )

    optimizer = torch.optim.AdamW(
        model.parameters(), lr=FINETUNE_LR, weight_decay=WEIGHT_DECAY
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=FINETUNE_EPOCHS, eta_min=1e-6
    )

    history = {
        'train_loss': [], 'train_llm': [], 'train_jepa': [],
        'val_loss': [], 'val_llm': [], 'val_jepa': [],
    }

    best_val_loss = float('inf')
    patience_ctr = 0
    best_state = None

    for epoch in range(1, FINETUNE_EPOCHS + 1):
        tr_loss, tr_llm, tr_jepa = finetune_one_epoch(
            model, finetune_loader, optimizer,
            lambda_jepa=lambda_jepa,
            jepa_dropout=JEPA_DROPOUT,
            device=device,
        )
        scheduler.step()

        vl_loss, vl_llm, vl_jepa = validate(
            model, val_loader, lambda_jepa=lambda_jepa, device=device
        )

        history['train_loss'].append(tr_loss)
        history['train_llm'].append(tr_llm)
        history['train_jepa'].append(tr_jepa)
        history['val_loss'].append(vl_loss)
        history['val_llm'].append(vl_llm)
        history['val_jepa'].append(vl_jepa)

        improved = ""
        if vl_loss < best_val_loss:
            best_val_loss = vl_loss
            patience_ctr = 0
            best_state = copy.deepcopy(model.state_dict())
            improved = " ★"
        else:
            patience_ctr += 1

        if epoch % 5 == 0 or epoch == 1 or improved:
            print(f"    Epoch {epoch:3d}/{FINETUNE_EPOCHS} | "
                  f"Train: {tr_loss:.4f} (LLM:{tr_llm:.4f} JEPA:{tr_jepa:.4f}) | "
                  f"Val: {vl_loss:.4f} (LLM:{vl_llm:.4f} JEPA:{vl_jepa:.4f}){improved}")

        if patience_ctr >= PATIENCE:
            print(f"\n    ⚠ Early stopping at epoch {epoch} "
                  f"(no improvement for {PATIENCE} epochs)")
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    save_path = f'./checkpoints/{model_name.lower()}_finetuned.pt'
    torch.save(model.state_dict(), save_path)
    print(f"\n  ✓ Fine-tuned model saved to {save_path}")
    print(f"  ✓ Best val loss: {best_val_loss:.4f}")
    print(f"  ✓ Stopped at epoch: {len(history['train_loss'])}")

    return history

Pretraining on QED

In [ ]:
qed_model = create_model(qed_vocab, device=device)

qed_pretrain_history = run_pretraining(
    model=qed_model,
    train_data=qed_train,
    vocab=qed_vocab,
    model_name='QED',
    device=device,
)

Pretraining on QCD

In [ ]:
qcd_model = create_model(qcd_vocab, device=device)

qcd_pretrain_history = run_pretraining(
    model=qcd_model,
    train_data=qcd_train,
    vocab=qcd_vocab,
    model_name='QCD',
    device=device,
)

Analysis of Pretraing of QCD and QED

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(qed_pretrain_history, 'b-o', markersize=3, linewidth=2)
axes[0].set_title('QED —  JEPA Pretraining', fontsize=13)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('JEPA Loss (Cosine Distance)')
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim(bottom=0)

axes[1].plot(qcd_pretrain_history, 'r-o', markersize=3, linewidth=2)
axes[1].set_title('QCD —  JEPA Pretraining', fontsize=13)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('JEPA Loss (Cosine Distance)')
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim(bottom=0)

plt.tight_layout()
plt.savefig('./checkpoints/phase1_pretraining.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"QED — Final pretrain JEPA loss: {qed_pretrain_history[-1]:.4f}")
print(f"QCD — Final pretrain JEPA loss: {qcd_pretrain_history[-1]:.4f}")

Finetuning QED

In [ ]:
# Now we fine-tune it with L_LLM + λ·L_JEPA on amplitude → squared amplitude.
qed_finetune_history = run_finetuning(
    model=qed_model,
    train_data=qed_train,
    val_data=qed_val,
    vocab=qed_vocab,
    lambda_jepa=LAMBDA_JEPA,
    model_name='QED',
    device=device,
)

In [ ]:
qed_results = run_evaluation(qed_model, qed_test, qed_vocab, 'QED', device)

In [ ]:
# Show Sample Predictions — See what the model generates
@torch.no_grad()
def show_predictions(model, test_data, vocab, n=10, max_len=300, device='cuda'):
    """
    Generate squared amplitudes from amplitudes and display them
    side by side with the ground truth.
    """
    model.eval()
    dataset = FinetuneDataset(test_data, vocab)

    correct = 0
    total = 0

    for i in range(min(n, len(dataset))):
        item = dataset[i]
        amp_ids = item['amp']
        sq_amp_ids = item['sq_amp']

        # Build input: <BOS> amp_tokens <SEP>
        amp_seq = amp_ids[amp_ids != vocab.pad_idx].tolist()
        if amp_seq[-1] == vocab.eos_idx:
            amp_seq = amp_seq[:-1]
        input_ids = amp_seq + [vocab.sep_idx]
        input_tensor = torch.tensor([input_ids], dtype=torch.long, device=device)

        # Greedy decode
        for _ in range(max_len):
            logits = model(input_tensor)
            next_token = logits[0, -1, :].argmax().item()
            input_tensor = torch.cat([
                input_tensor,
                torch.tensor([[next_token]], device=device)
            ], dim=1)
            if next_token == vocab.eos_idx:
                break

        # Extract generated tokens (after <SEP>)
        full_gen = input_tensor[0].tolist()
        try:
            sep_loc = full_gen.index(vocab.sep_idx)
        except ValueError:
            sep_loc = 0

        gen_ids = []
        for t in full_gen[sep_loc + 1:]:
            if t == vocab.eos_idx or t == vocab.pad_idx:
                break
            gen_ids.append(t)

        # Extract ground truth (from sq_amp)
        gt_ids = []
        for t in sq_amp_ids.tolist():
            if t in (vocab.bos_idx, vocab.pad_idx):
                continue
            if t == vocab.eos_idx:
                break
            gt_ids.append(t)

        # Decode to readable text
        gen_text = vocab.decode(gen_ids)
        gt_text = vocab.decode(gt_ids)
        amp_text = vocab.decode([t for t in amp_seq if t not in (vocab.bos_idx,)])

        is_match = gen_ids == gt_ids
        if is_match:
            correct += 1
        total += 1

        match_str = "✓ EXACT MATCH" if is_match else "✗ MISMATCH"

        print(f"\n{'━'*80}")
        print(f"  Sample {i+1} — {match_str}")
        print(f"{'━'*80}")
        print(f"  INPUT (amplitude):")
        print(f"    {amp_text[:120]}")
        if len(amp_text) > 120:
            print(f"    {amp_text[120:240]}")
        print(f"\n  TARGET (squared amplitude):")
        print(f"    {gt_text[:120]}")
        if len(gt_text) > 120:
            print(f"    {gt_text[120:240]}")
        print(f"\n  GENERATED:")
        print(f"    {gen_text[:120]}")
        if len(gen_text) > 120:
            print(f"    {gen_text[120:240]}")

        # Show where they differ (first difference)
        if not is_match:
            min_len = min(len(gen_ids), len(gt_ids))
            diff_pos = -1
            for j in range(min_len):
                if gen_ids[j] != gt_ids[j]:
                    diff_pos = j
                    break
            if diff_pos == -1 and len(gen_ids) != len(gt_ids):
                diff_pos = min_len

            if diff_pos >= 0:
                gt_tok = vocab.decode([gt_ids[diff_pos]]) if diff_pos < len(gt_ids) else "<END>"
                gen_tok = vocab.decode([gen_ids[diff_pos]]) if diff_pos < len(gen_ids) else "<END>"
                print(f"\n  FIRST DIFFERENCE at token {diff_pos}:")
                print(f"    Expected: '{gt_tok}' | Got: '{gen_tok}'")
                print(f"    Generated length: {len(gen_ids)} | Target length: {len(gt_ids)}")

    print(f"\n{'━'*80}")
    print(f"  SUMMARY: {correct}/{total} exact matches ({correct/max(total,1)*100:.1f}%)")
    print(f"{'━'*80}")


# ---- Run it for QED ----
print("="*80)
print("  QED — SAMPLE PREDICTIONS")
print("="*80)
show_predictions(qed_model, qed_test, qed_vocab, n=10, device=device)

Finetuning QCD

In [ ]:
qcd_finetune_history = run_finetuning(
    model=qcd_model,
    train_data=qcd_train,
    val_data=qcd_val,
    vocab=qcd_vocab,
    lambda_jepa=LAMBDA_JEPA,
    model_name='QCD',
    device=device,
)

In [ ]:
# =============================================================================
# Step 3: ONLY if batch_size=8 still crashes
# Uses gradient accumulation: batch_size=4, accumulate over 8 steps = effective 32
# =============================================================================

def finetune_one_epoch_grad_accum(model, loader, optimizer, lambda_jepa,
                                   accum_steps=8, jepa_dropout=0.5, device='cuda'):
    """Finetune with gradient accumulation to handle OOM."""
    model.train()
    total_loss = 0.0
    total_llm = 0.0
    total_jepa = 0.0
    n_batches = 0

    optimizer.zero_grad()

    for step, batch in enumerate(loader):
        full = batch['full'].to(device)
        sep_pos = batch['sep_pos'].to(device)
        amp = batch['amp'].to(device)
        sq_amp = batch['sq_amp'].to(device)

        # L_LLM
        logits = model(full)
        loss_llm = compute_llm_loss(logits, full, sep_pos, model.pad_idx)

        # L_JEPA
        batch_size = amp.size(0)
        n_jepa = max(1, int(batch_size * (1 - jepa_dropout)))
        jepa_idx = torch.randperm(batch_size, device=device)[:n_jepa]

        pred_emb = model.predict_embedding(amp[jepa_idx])
        with torch.no_grad():
            target_emb = model.get_embedding(sq_amp[jepa_idx])
        loss_jepa = compute_jepa_loss(pred_emb, target_emb)

        loss = (loss_llm + lambda_jepa * loss_jepa) / accum_steps
        loss.backward()

        if (step + 1) % accum_steps == 0 or (step + 1) == len(loader):
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            optimizer.zero_grad()

        total_loss += (loss.item() * accum_steps)
        total_llm += loss_llm.item()
        total_jepa += loss_jepa.item()
        n_batches += 1

    n = max(n_batches, 1)
    return total_loss / n, total_llm / n, total_jepa / n


# Use batch_size=4 with gradient accumulation
torch.cuda.empty_cache()

finetune_loader = DataLoader(
    FinetuneDataset(qcd_train, qcd_vocab),
    batch_size=4, shuffle=True, collate_fn=collate_finetune,
)
val_loader = DataLoader(
    FinetuneDataset(qcd_val, qcd_vocab),
    batch_size=4, shuffle=False, collate_fn=collate_finetune,
)

optimizer = torch.optim.AdamW(qcd_model.parameters(), lr=FINETUNE_LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=FINETUNE_EPOCHS, eta_min=1e-6)

qcd_finetune_history = {
    'train_loss': [], 'train_llm': [], 'train_jepa': [],
    'val_loss': [], 'val_llm': [], 'val_jepa': [],
}

best_val_loss = float('inf')
patience_ctr = 0
best_state = None

print("QCD Fine-tuning with gradient accumulation (batch=4, accum=8)")
for epoch in range(1, FINETUNE_EPOCHS + 1):
    tr_loss, tr_llm, tr_jepa = finetune_one_epoch_grad_accum(
        qcd_model, finetune_loader, optimizer,
        lambda_jepa=LAMBDA_JEPA, accum_steps=8,
        jepa_dropout=JEPA_DROPOUT, device=device,
    )
    scheduler.step()

    vl_loss, vl_llm, vl_jepa = validate(
        qcd_model, val_loader, lambda_jepa=LAMBDA_JEPA, device=device
    )

    qcd_finetune_history['train_loss'].append(tr_loss)
    qcd_finetune_history['train_llm'].append(tr_llm)
    qcd_finetune_history['train_jepa'].append(tr_jepa)
    qcd_finetune_history['val_loss'].append(vl_loss)
    qcd_finetune_history['val_llm'].append(vl_llm)
    qcd_finetune_history['val_jepa'].append(vl_jepa)

    improved = ""
    if vl_loss < best_val_loss:
        best_val_loss = vl_loss
        patience_ctr = 0
        best_state = copy.deepcopy(qcd_model.state_dict())
        improved = " ★"
    else:
        patience_ctr += 1

    if epoch % 5 == 0 or epoch == 1 or improved:
        print(f"  Epoch {epoch:3d}/{FINETUNE_EPOCHS} | "
              f"Train: {tr_loss:.4f} (LLM:{tr_llm:.4f} JEPA:{tr_jepa:.4f}) | "
              f"Val: {vl_loss:.4f} (LLM:{vl_llm:.4f} JEPA:{vl_jepa:.4f}){improved}")

    if patience_ctr >= PATIENCE:
        print(f"  Early stopping at epoch {epoch}")
        break

if best_state:
    qcd_model.load_state_dict(best_state)
torch.save(qcd_model.state_dict(), './checkpoints/qcd_finetuned.pt')
print(f"✓ QCD finetuned model saved (best val: {best_val_loss:.4f})")

QCD Fine-tuning with gradient accumulation (batch=4, accum=8)
  Epoch   1/50 | Train: 5.0065 (LLM:4.9830 JEPA:0.0235) | Val: 4.6810 (LLM:4.6776 JEPA:0.0034) ★
  Epoch   2/50 | Train: 4.5336 (LLM:4.5088 JEPA:0.0248) | Val: 4.2416 (LLM:4.2377 JEPA:0.0038) ★
  Epoch   3/50 | Train: 4.1359 (LLM:4.1086 JEPA:0.0273) | Val: 3.8956 (LLM:3.8904 JEPA:0.0052) ★
  Epoch   4/50 | Train: 3.8485 (LLM:3.8157 JEPA:0.0328) | Val: 3.6756 (LLM:3.6686 JEPA:0.0070) ★
  Epoch   5/50 | Train: 3.6577 (LLM:3.6236 JEPA:0.0341) | Val: 3.5192 (LLM:3.5137 JEPA:0.0056) ★
  Epoch   6/50 | Train: 3.5156 (LLM:3.4843 JEPA:0.0312) | Val: 3.3997 (LLM:3.3949 JEPA:0.0047) ★
  Epoch   7/50 | Train: 3.3983 (LLM:3.3688 JEPA:0.0295) | Val: 3.2940 (LLM:3.2892 JEPA:0.0048) ★
  Epoch   8/50 | Train: 3.2949 (LLM:3.2658 JEPA:0.0292) | Val: 3.1976 (LLM:3.1924 JEPA:0.0052) ★
  Epoch   9/50 | Train: 3.1991 (LLM:3.1691 JEPA:0.0300) | Val: 3.0964 (LLM:3.0896 JEPA:0.0067) ★
  Epoch  10/50 | Train: 3.0923 (LLM:3.0571 JEPA:0.0352) | Val: 2.

In [ ]:
# Show Sample Predictions — See what the model generates
@torch.no_grad()
def show_predictions(model, test_data, vocab, n=10, max_len=300, device='cuda'):
    """
    Generate squared amplitudes from amplitudes and display them
    side by side with the ground truth.
    """
    model.eval()
    dataset = FinetuneDataset(test_data, vocab)

    correct = 0
    total = 0

    for i in range(min(n, len(dataset))):
        item = dataset[i]
        amp_ids = item['amp']
        sq_amp_ids = item['sq_amp']

        # Build input: <BOS> amp_tokens <SEP>
        amp_seq = amp_ids[amp_ids != vocab.pad_idx].tolist()
        if amp_seq[-1] == vocab.eos_idx:
            amp_seq = amp_seq[:-1]
        input_ids = amp_seq + [vocab.sep_idx]
        input_tensor = torch.tensor([input_ids], dtype=torch.long, device=device)

        # Greedy decode
        for _ in range(max_len):
            logits = model(input_tensor)
            next_token = logits[0, -1, :].argmax().item()
            input_tensor = torch.cat([
                input_tensor,
                torch.tensor([[next_token]], device=device)
            ], dim=1)
            if next_token == vocab.eos_idx:
                break

        # Extract generated tokens (after <SEP>)
        full_gen = input_tensor[0].tolist()
        try:
            sep_loc = full_gen.index(vocab.sep_idx)
        except ValueError:
            sep_loc = 0

        gen_ids = []
        for t in full_gen[sep_loc + 1:]:
            if t == vocab.eos_idx or t == vocab.pad_idx:
                break
            gen_ids.append(t)

        # Extract ground truth (from sq_amp)
        gt_ids = []
        for t in sq_amp_ids.tolist():
            if t in (vocab.bos_idx, vocab.pad_idx):
                continue
            if t == vocab.eos_idx:
                break
            gt_ids.append(t)

        # Decode to readable text
        gen_text = vocab.decode(gen_ids)
        gt_text = vocab.decode(gt_ids)
        amp_text = vocab.decode([t for t in amp_seq if t not in (vocab.bos_idx,)])

        is_match = gen_ids == gt_ids
        if is_match:
            correct += 1
        total += 1

        match_str = "✓ EXACT MATCH" if is_match else "✗ MISMATCH"

        print(f"\n{'━'*80}")
        print(f"  Sample {i+1} — {match_str}")
        print(f"{'━'*80}")
        print(f"  INPUT (amplitude):")
        print(f"    {amp_text[:120]}")
        if len(amp_text) > 120:
            print(f"    {amp_text[120:240]}")
        print(f"\n  TARGET (squared amplitude):")
        print(f"    {gt_text[:120]}")
        if len(gt_text) > 120:
            print(f"    {gt_text[120:240]}")
        print(f"\n  GENERATED:")
        print(f"    {gen_text[:120]}")
        if len(gen_text) > 120:
            print(f"    {gen_text[120:240]}")

        # Show where they differ (first difference)
        if not is_match:
            min_len = min(len(gen_ids), len(gt_ids))
            diff_pos = -1
            for j in range(min_len):
                if gen_ids[j] != gt_ids[j]:
                    diff_pos = j
                    break
            if diff_pos == -1 and len(gen_ids) != len(gt_ids):
                diff_pos = min_len

            if diff_pos >= 0:
                gt_tok = vocab.decode([gt_ids[diff_pos]]) if diff_pos < len(gt_ids) else "<END>"
                gen_tok = vocab.decode([gen_ids[diff_pos]]) if diff_pos < len(gen_ids) else "<END>"
                print(f"\n  FIRST DIFFERENCE at token {diff_pos}:")
                print(f"    Expected: '{gt_tok}' | Got: '{gen_tok}'")
                print(f"    Generated length: {len(gen_ids)} | Target length: {len(gt_ids)}")

    print(f"\n{'━'*80}")
    print(f"  SUMMARY: {correct}/{total} exact matches ({correct/max(total,1)*100:.1f}%)")
    print(f"{'━'*80}")


print("  QCD — SAMPLE PREDICTIONS")
print("="*80)

show_predictions(qcd_model, qcd_test, qcd_vocab, n=10, device=device)

  QCD — SAMPLE PREDICTIONS

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Sample 1 — ✗ MISMATCH
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  INPUT (amplitude):
    1 / 2 * i * g ^ 2 * ( MOMENTUM_2_ % \ INDEX_0 * gamma_{ + % \ INDEX_0 , % INDEX_1 , % INDEX_2 } * gamma_{ % \ INDEX_3 , 
    % INDEX_4 , % INDEX_1 } * gamma_{ % \ INDEX_5 , % INDEX_2 , % INDEX_6 } * T_C_10_{ % INDEX_7 , % INDEX_8 , % INDEX_9 } *

  TARGET (squared amplitude):
    1 / 6 * g ^ 4 * MOMENTUM_23 * MOMENTUM_24 * ( MOMENTUM_13 + - 1 / 2 * reg_prop ) ^ ( - 2 ) + - 1 / 6 * i * g ^ 2 * ( i *
     g ^ 2 * m_u ^ 2 * ( MOMENTUM_23 + 2 * MOMENTUM_24 ) / ( MOMENTUM_13 + - 1 / 2 * reg_prop ) + ( - 2 ) * i * g ^ 2 * m_u 

  GENERATED:
    4 / 2 * g ^ 2 * g ^ 2 * g ^ 2 * g ^ 2 * ( - 2 * ( - 2 * ( - 1 / 2 * g ^ 2 * g ^ 2 * g ^ 2 * g ^ 2 * ( - 1 / 2 * reg_prop
     ) / 2 * reg_prop ) / 2 * reg_prop ) / 2 * reg_prop ) + - 1 / 2 * g ^ 2 * reg_prop ) + - 1 

Evaluating both the models on test sets

In [ ]:
qed_results = run_evaluation(qed_model, qed_test, qed_vocab, 'QED', device)
qcd_results = run_evaluation(qcd_model, qcd_test, qcd_vocab, 'QCD', device)